# Practice Lab: Scaling AI Applications (Lesson 12.7)

Module 12 . 4 exercises . Concurrency + cold starts + queue metrics + GPU costs


# Lesson 12.7: Scaling AI Applications

4 exercises: 2E + 1M + 1C

All exercises simulate scaling (no cloud infra required).


In [ ]:
import random, numpy as np
random.seed(42); np.random.seed(42)


---
## Exercise 1 (Easy): Concurrency Load Test


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, numpy as np
random.seed(42); np.random.seed(42)

class CRSim:
    def __init__(self,conc,max_inst=20,req_ms=800):
        self.conc=conc; self.max=max_inst; self.req_ms=req_ms
    def test(self,total=200,concurrent=50):
        inst=min(self.max,max(1,concurrent//self.conc+(1 if concurrent%self.conc else 0)))
        lats=[]; errs=0
        for _ in range(total):
            qd=max(0,(concurrent/(inst*self.conc)-1))*self.req_ms
            lat=self.req_ms+qd+abs(random.gauss(0,self.req_ms*0.15))
            if lat>self.req_ms*5: errs+=1; lat=self.req_ms*5
            lats.append(lat)
        lats.sort()
        return {"inst":inst,"avg":np.mean(lats),"p95":np.percentile(lats,95),"err":errs/total*100}

print("Concurrency Load Test:")
print(f"  {'Config':<30} {'Inst':>5} {'Avg':>7} {'P95':>7} {'Err%':>5}")
for name,c,ms in [("conc=10 (LLM-heavy)",10,800),("conc=80 (cache)",80,100),
    ("conc=10 (cache)",10,100),("conc=80 (LLM)",80,800)]:
    r=CRSim(c,req_ms=ms).test()
    print(f"  {name:<30} {r['inst']:>5} {r['avg']:>6.0f}ms {r['p95']:>6.0f}ms {r['err']:>4.0f}%")
print(f"\n  Optimal: LLM=conc 10-20 | Cache=conc 80+")


</details>


---
## Exercise 2 (Easy): Cold Start Measurement


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class ColdSim:
    def __init__(self,min_inst,cold_ms=500,warm_ms=200):
        self.min_inst=min_inst; self.cold=cold_ms; self.warm=warm_ms; self.cost=min_inst*5.0
    def run(self,n=20):
        lats=[]
        for i in range(n):
            if i<1 and self.min_inst==0: lat=self.cold+self.warm+random.gauss(0,50)
            elif i<2 and self.min_inst==0: lat=self.warm+150+random.gauss(0,30)
            else: lat=self.warm+random.gauss(0,20)
            lats.append(max(50,lat))
        ls=sorted(lats)
        return {"first":lats[0],"avg":sum(lats)/n,"p99":ls[-1],"cold":1 if self.min_inst==0 else 0,"cost":self.cost}

print("Cold Start Measurement:")
print(f"  {'Config':<33} {'1st':>7} {'Avg':>7} {'P99':>7} {'Cold':>5} {'$/mo':>6}")
for name,mi in [("min=0 (scale-to-zero)",0),("min=1 (always warm)",1),("min=2 (HA)",2)]:
    r=ColdSim(mi).run()
    print(f"  {name:<33} {r['first']:>6.0f}ms {r['avg']:>6.0f}ms {r['p99']:>6.0f}ms {r['cold']:>5} ${r['cost']:>5.0f}")
print(f"\n  Dev: min=0 | Prod: min=1 ($5/mo) | HA: min=2 ($10/mo)")


</details>


---
## Exercise 3 (Medium): Queue Metrics Dashboard


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class QDash:
    def __init__(self,workers=5,job_ms=800): self.w=workers; self.jms=job_ms; self.snaps=[]
    def run(self,total=100):
        q=list(range(total)); proc=[]; done=0; t=0; waits=[]
        snap_at=[0,20,40,60,80]
        si=0
        while q or proc:
            while q and len(proc)<self.w:
                q.pop(0); waits.append(t); proc.append(t+self.jms+random.randint(-100,200))
            if si<len(snap_at) and done>=snap_at[si]:
                tp=done/max(t/1000,0.001); aw=sum(waits)/max(len(waits),1)
                self.snaps.append({"t":round(t/1000,1),"q":len(q),"p":len(proc),"d":done,
                    "w":round(aw),"rps":round(tp,1),"act":"UP" if len(q)>self.w*3 else "OK"})
                si+=1
            t+=100
            fin=[p for p in proc if p<=t]
            for f in fin: proc.remove(f); done+=1
        self.snaps.append({"t":round(t/1000,1),"q":0,"p":0,"d":done,"w":round(sum(waits)/max(len(waits),1)),
            "rps":round(done/(t/1000),1),"act":"DOWN"})

d=QDash(5,800); d.run(100)
print("Queue Metrics Dashboard:")
print(f"  {'Time':>6} {'Queue':>6} {'Active':>7} {'Done':>6} {'Wait':>7} {'RPS':>5} {'Action':>7}")
for s in d.snaps:
    print(f"  {s['t']:>5.1f}s {s['q']:>6} {s['p']:>7} {s['d']:>6} {s['w']:>6}ms {s['rps']:>5.1f} {s['act']:>7}")
print(f"\n  Scale UP: queue > {d.w*3} | Scale DOWN: idle 5min")


</details>


---
## Exercise 4 (Challenge): GPU Cost Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class CostModel:
    def __init__(self):
        self.api={"flash":{"c":0.00015,"t":500},"pro":{"c":0.00125,"t":500}}
        self.gpu={"GCP T4":{"hr":0.35,"tps":80},"GCP A100":{"hr":2.50,"tps":400},
            "E2E T4":{"hr":0.14,"tps":80},"E2E H100":{"hr":3.50,"tps":800}}
    def api_cost(self,m,reqs): c=self.api[m]; return reqs*c["t"]/1000*c["c"]
    def gpu_cost(self,g,reqs):
        c=self.gpu[g]; hrs=max(reqs*500/(c["tps"]*3600*0.6),720); return hrs*c["hr"]*1.2
    def crossover(self,api,gpu):
        for r in range(50000,5000001,50000):
            if self.gpu_cost(gpu,r)<self.api_cost(api,r): return r
        return None

m=CostModel()
print("GPU Scaling Cost Comparison:")
print(f"  {'':>22} {'100K':>8} {'300K':>8} {'500K':>8} {'1M':>8}")
for api in ["flash","pro"]:
    costs=[m.api_cost(api,s) for s in [100000,300000,500000,1000000]]
    print(f"  {'gemini-'+api:<22}"+"".join(f" ${c:>7.0f}" for c in costs))
for gpu in ["GCP T4","GCP A100","E2E T4","E2E H100"]:
    costs=[m.gpu_cost(gpu,s) for s in [100000,300000,500000,1000000]]
    print(f"  {gpu:<22}"+"".join(f" ${c:>7.0f}" for c in costs))
print(f"\n  Crossover:")
for api in ["flash","pro"]:
    for gpu in ["GCP T4","E2E T4"]:
        x=m.crossover(api,gpu)
        print(f"    {api} vs {gpu}: {f'{x:,} req/mo' if x else 'API always cheaper'}")
print(f"  Rule: <300K=API | 300K-1M=E2E India | >1M=self-host")


</details>
